In [7]:
from scipy.io import loadmat
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [6]:
data = np.load('../data/ersst_anomaly.npy')
print(data.shape)


(900, 45, 180)


In [10]:
def coarse_grain(data, factor=5):
    time_steps, lat, lon = data.shape
    new_lat = lat // factor
    new_lon = lon // factor
    
    # Reshape to create sub-grids and average
    reshaped = data.reshape(time_steps, new_lat, factor, new_lon, factor)
    coarse_grained = reshaped.mean(axis=(2, 4))
    return coarse_grained

In [13]:
def create_comparison_animation(original_data, coarse_data, output_path='sst_comparison.mp4', fps=10):
    """
    Create a side-by-side animation of grids and their time series
    """
    # Calculate spatial means for time series
    original_means = original_data.mean(axis=(1,2))
    coarse_means = coarse_data.mean(axis=(1,2))
    
    # Create figure with subplots: 2x2 grid
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(2, 2, height_ratios=[2, 1])
    
    # Grid visualization subplots
    ax1 = fig.add_subplot(gs[0, 0])  # top left
    ax2 = fig.add_subplot(gs[0, 1])  # top right
    
    # Time series subplots
    ax3 = fig.add_subplot(gs[1, 0])  # bottom left
    ax4 = fig.add_subplot(gs[1, 1])  # bottom right
    
    # Calculate global min and max for consistent colormap
    vmin = min(original_data.min(), coarse_data.min())
    vmax = max(original_data.max(), coarse_data.max())
    
    # Initialize grid plots
    im1 = ax1.imshow(original_data[0], 
                     cmap='RdBu_r',
                     aspect='auto',
                     vmin=vmin,
                     vmax=vmax)
    im2 = ax2.imshow(coarse_data[0], 
                     cmap='RdBu_r',
                     aspect='auto',
                     vmin=vmin,
                     vmax=vmax)
    
    # Add colorbars
    plt.colorbar(im1, ax=ax1, label='Temperature Anomaly (°C)')
    plt.colorbar(im2, ax=ax2, label='Temperature Anomaly (°C)')
    
    # Set titles
    ax1.set_title(f'Original Grid ({original_data.shape[1]}x{original_data.shape[2]})')
    ax2.set_title(f'Coarse Grid ({coarse_data.shape[1]}x{coarse_data.shape[2]})')
    
    # Initialize time series plots
    time_points = np.arange(len(original_means))
    line1, = ax3.plot(time_points[0:1], original_means[0:1], 'b-')
    line2, = ax4.plot(time_points[0:1], coarse_means[0:1], 'r-')
    
    # Set time series plot properties
    for ax, title in [(ax3, 'Original Grid Mean'), (ax4, 'Coarse Grid Mean')]:
        ax.set_xlim(0, len(time_points))
        ax.set_ylim(min(original_means.min(), coarse_means.min()),
                   max(original_means.max(), coarse_means.max()))
        ax.set_title(title)
        ax.set_xlabel('Time Step')
        ax.set_ylabel('Spatial Mean Temperature')
        ax.grid(True)
    
    # Add time step counter
    time_text = fig.text(0.5, 0.95, '', ha='center')
    
    def update(frame):
        """Update function for animation"""
        # Update grid plots
        im1.set_array(original_data[frame])
        im2.set_array(coarse_data[frame])
        
        # Update time series (show up to current frame)
        line1.set_data(time_points[:frame+1], original_means[:frame+1])
        line2.set_data(time_points[:frame+1], coarse_means[:frame+1])
        
        # Update time counter
        time_text.set_text(f'Time step: {frame}')
        
        return im1, im2, line1, line2, time_text
    
    # Create animation
    anim = animation.FuncAnimation(fig, 
                                 update, 
                                 frames=len(original_data),
                                 interval=1000/fps,
                                 blit=True)
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    
    # Save animation
    anim.save(output_path, writer='ffmpeg', fps=fps)
    plt.close()


In [14]:
coarse_data = coarse_grain(data, factor=5)
print(f"Coarse data shape: {coarse_data.shape}")

# Create comparison animation
create_comparison_animation(data, 
                            coarse_data,
                            output_path='sst_comparison_with_ts.mp4', 
                            fps=10)

Coarse data shape: (900, 9, 36)


In [19]:
import torch
def batch_data_to_timeseries(batched_ts):
    if isinstance(batched_ts, torch.Tensor): 
        batched_ts = batched_ts.detach().cpu().numpy()

    if len(batched_ts.shape) == 4:
        batched_ts = batched_ts.squeeze(1)

    if len(batched_ts) == 1:
        return batched_ts.squeeze().T 
    # (1, t, n)
    time_series = batched_ts[0, :, :].T.copy()  # Start with the first window
    print(f"time_series.shape: {time_series.shape}")
    # Add one new point from each subsequent window
    for i in range(1, len(batched_ts)):
        # each appending is: (1, n)
        time_series = np.concatenate((time_series, np.expand_dims(batched_ts[i, :, -1],axis=0)),axis=0)
    # (T,n)
    return time_series

In [ ]:
x_example = torch.randn(12, 324, 182) # (t, n, b) 
batch_data_to_timeseries(x_example).shape

time_series.shape: (182, 324)


(193, 324)